# VLS comparison between two runs

Loads two raw FLASH runs (config 2), applies the same preprocessing
to each, and produces side-by-side / overlay comparisons:

1. Average spectrum and per-bunch / per-train maps (one panel per run).
2. Per-shot spectral moments and the distributions of COM and width.
3. GMD vs integrated VLS, GMD-binned and COM-binned average spectra.
4. Final comparison: spectra per (COM bin x GMD bin), GMD-normalised
   via `sum(spectra) / sum(GMD)` rather than `mean(spec / gmd)`.
5. Mean `|FFT(spectrum)|` per run for shot-to-shot interference fringes
   whose phase washes out of the mean spectrum.

Background is read from a separate "shutter-closed" run and subtracted
identically from every run.

In [ ]:
import sys
from pathlib import Path
_REPO_ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(_REPO_ROOT / "analysis" / "scripts"))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

import config
from data_loading import load_raw_h5
from binning import arb_bool_ar, bin_and_average, bin_and_sum_ratio
%matplotlib inline

## Parameters

In [ ]:
# Raw runs to compare. Add or remove run numbers freely - every
# downstream cell loops over `runs` so adding a third run just
# adds a panel / extra line.
RUN_NOS    = [58815, 58816]
BG_RUN_NO  = 58814
MAX_FILES  = 20
TRAIN_LENGTH = 110

# Bunch-axis roll (np.roll) applied to align the Gotthard line index
# with the FEL bunch index.
GOTTHARD_ROLL = -2

# VLS pixel ROI (after roll). None keeps the full pixel axis.
ROI = (560, 615)

# Half-open bunch range used for the per-train auto background.
BG_BUNCH_RANGE = (100, 108)

# Half-open bunch range kept after baseline subtraction.
SIGNAL_BUNCH_RANGE = (0, 101)

# Log-stretch the bunch / train maps.
LOG_MAP = False

print(f"runs to compare : {RUN_NOS}")
print(f"background run  : {BG_RUN_NO}")
print(f"max files / run : {MAX_FILES}")
print(f"ROI             : {ROI}")
print(f"gotthard roll   : {GOTTHARD_ROLL}")

## Background (shutter-closed run)

The closed-shutter run is loaded once, rolled to the same bunch
frame as the data, and averaged over trains to give a per-bunch
background spectrum. Every comparison run subtracts this same
background.

In [ ]:
vls_bg = (
    load_raw_h5(BG_RUN_NO, config=2, max_files=MAX_FILES,
                train_length=TRAIN_LENGTH)
    .roll_vls_bunches(GOTTHARD_ROLL)
    .vls.mean(0)
)
print(f"background source: run {BG_RUN_NO}")
print(f"background shape : {vls_bg.shape}")

fig, ax = plt.subplots(figsize=(9, 4))
norm = (mcolors.LogNorm(vmin=max(np.nanmin(vls_bg), 1e-3),
                        vmax=np.nanmax(vls_bg))
        if LOG_MAP else None)
im = ax.pcolormesh(vls_bg, cmap="inferno", shading="auto", norm=norm)
fig.colorbar(im, ax=ax, label="Mean intensity (arb.)")
ax.set_xlabel("Pixel")
ax.set_ylabel("Bunch index")
ax.set_title(f"Closed-shutter background (run {BG_RUN_NO})")
fig.tight_layout()
plt.show()

## Load and prepare each run

Identical preprocessing for every run: roll -> subtract the closed-
shutter background -> crop to ROI -> per-train auto bg from the
trailing bunches -> trim to the signal bunches -> compute per-shot
spectral moments. The results live in a `runs` dict keyed by run
number; every downstream cell iterates over it.

In [ ]:
def load_and_prepare(run_no, bg_spec):
    """Raw load -> roll -> bg-run subtract -> crop -> auto-bg -> trim -> moments."""
    data = load_raw_h5(run_no, config=2, max_files=MAX_FILES,
                       train_length=TRAIN_LENGTH)
    data = data.roll_vls_bunches(GOTTHARD_ROLL)
    data = data.subtract_background(bg_spec)
    if ROI is not None:
        data = data.crop_vls(*ROI)
    data = data.auto_subtract_background(BG_BUNCH_RANGE)
    data = data.trim_bunches(*SIGNAL_BUNCH_RANGE)
    data = data.compute_vls_moments()
    return data


runs = {r: load_and_prepare(r, vls_bg) for r in RUN_NOS}

for run_no, data in runs.items():
    n_t, n_b, n_p = data.vls.shape
    print(f"run {run_no}: vls shape={data.vls.shape}, "
          f"crop={data.vls_crop_roi}, bg={data.vls_background_roi}, "
          f"shots={n_t * n_b}")

## Average spectrum per run

Overlay of the all-shot mean spectrum from each run. Shaded band is
+/- 1 std across shots.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for run_no, data in runs.items():
    flat = data.vls.reshape(-1, data.vls.shape[-1])
    mean_spec = np.nanmean(flat, axis=0)
    std_spec  = np.nanstd(flat,  axis=0)
    line, = ax.plot(data.vls_pixels, mean_spec, lw=1.4,
                    label=f"run {run_no}  ({flat.shape[0]} shots)")
    ax.fill_between(data.vls_pixels,
                    mean_spec - std_spec, mean_spec + std_spec,
                    color=line.get_color(), alpha=0.18)
ax.set_xlabel("Pixel")
ax.set_ylabel("Intensity (arb.)")
ax.set_title("Average VLS spectrum  (shaded = +/- 1 std)")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

## Mean spectrum vs bunch index

In [ ]:
fig, axes = plt.subplots(1, len(runs),
                         figsize=(6 * len(runs), 5),
                         sharex=True, sharey=True)
axes = np.atleast_1d(axes)
for ax, (run_no, data) in zip(axes, runs.items()):
    mean_by_bunch = np.nanmean(data.vls, axis=0)
    norm = (mcolors.LogNorm(vmin=max(np.nanmin(mean_by_bunch), 1e-3),
                            vmax=np.nanmax(mean_by_bunch))
            if LOG_MAP else None)
    im = ax.pcolormesh(data.vls_pixels, np.arange(data.vls.shape[1]),
                       mean_by_bunch, cmap="inferno",
                       shading="auto", norm=norm)
    fig.colorbar(im, ax=ax, label="Mean intensity (arb.)")
    ax.set_xlabel("Pixel")
    ax.set_ylabel("Bunch index")
    ax.set_title(f"run {run_no}")
fig.suptitle("Mean spectrum vs bunch index")
fig.tight_layout()
plt.show()

## Mean spectrum vs train index

In [ ]:
fig, axes = plt.subplots(1, len(runs),
                         figsize=(6 * len(runs), 5), sharex=True)
axes = np.atleast_1d(axes)
for ax, (run_no, data) in zip(axes, runs.items()):
    mean_by_train = np.nanmean(data.vls, axis=1)
    norm = (mcolors.LogNorm(vmin=max(np.nanmin(mean_by_train), 1e-3),
                            vmax=np.nanmax(mean_by_train))
            if LOG_MAP else None)
    im = ax.pcolormesh(data.vls_pixels, data.tID, mean_by_train,
                       cmap="inferno", shading="auto", norm=norm)
    fig.colorbar(im, ax=ax, label="Mean intensity (arb.)")
    ax.set_xlabel("Pixel")
    ax.set_ylabel("Train ID")
    ax.set_title(f"run {run_no}")
fig.suptitle("Mean spectrum vs train index")
fig.tight_layout()
plt.show()

## Per-shot spectral moments

Per-shot sum / centre-of-mass / width on the cropped, background-
subtracted spectra. Each column is one moment; one row per run.
Histograms are clipped to the 1-99 %ile so a handful of noise
blow-ups don't dominate.

In [ ]:
labels = ("Sum (arb.)", "Centre of mass (pixel)", "Width (pixel)")
colours = ("mediumseagreen", "steelblue", "indianred")

fig, axes = plt.subplots(len(runs), 3,
                         figsize=(15, 4 * len(runs)),
                         sharex="col")
axes = np.atleast_2d(axes)

for row, (run_no, data) in zip(axes, runs.items()):
    moments = (data.vls_sums, data.vls_coms, data.vls_widths)
    for ax, vals, lab, col in zip(row, moments, labels, colours):
        finite = vals.ravel()[np.isfinite(vals.ravel())]
        lo, hi = np.nanpercentile(finite, [1, 99])
        ax.hist(np.clip(finite, lo, hi), bins=120,
                color=col, edgecolor="black", linewidth=0.3)
        ax.set_xlabel(lab)
        ax.set_xlim(lo, hi)
        ax.grid(alpha=0.3)
    row[0].set_ylabel(f"run {run_no}\nshots")

fig.suptitle("Per-shot spectral moments  (clipped to 1-99 %ile)")
fig.tight_layout()
plt.show()

## COM and width distributions: run-vs-run overlay

Direct overlay of the per-shot COM and width distributions for the
two runs. Histograms are density-normalised and share a joint 1-99
%ile clip so differences in the bulk are visually directly
comparable.

In [ ]:
def _shared_clip(arrays, lo_pc=1, hi_pc=99):
    finite = np.concatenate([a.ravel()[np.isfinite(a.ravel())] for a in arrays])
    return tuple(np.nanpercentile(finite, [lo_pc, hi_pc]))


com_clip   = _shared_clip([d.vls_coms   for d in runs.values()])
width_clip = _shared_clip([d.vls_widths for d in runs.values()])

fig, (ax_com, ax_w) = plt.subplots(1, 2, figsize=(12, 4))
for run_no, data in runs.items():
    com = data.vls_coms.ravel()
    w   = data.vls_widths.ravel()
    com = com[np.isfinite(com)]
    w   = w[np.isfinite(w)]
    ax_com.hist(np.clip(com, *com_clip),   bins=120, alpha=0.5,
                label=f"run {run_no}", density=True)
    ax_w.hist(  np.clip(w,   *width_clip), bins=120, alpha=0.5,
                label=f"run {run_no}", density=True)

ax_com.set_xlabel("Centre of mass (pixel)")
ax_com.set_ylabel("Density")
ax_com.set_xlim(com_clip)
ax_com.grid(alpha=0.3)
ax_com.legend()

ax_w.set_xlabel("Width (pixel)")
ax_w.set_ylabel("Density")
ax_w.set_xlim(width_clip)
ax_w.grid(alpha=0.3)
ax_w.legend()

fig.suptitle("Per-shot moment distributions (density, joint 1-99 %ile clip)")
fig.tight_layout()
plt.show()

## GMD vs per-shot VLS sum

One panel per run. The 2D hist shows the joint distribution, the
white error bars show the GMD-percentile-binned mean +/- std, and
the dashed line is the OLS linear fit over all shots.

In [ ]:
N_GMD_BINS = 10

fig, axes = plt.subplots(1, len(runs), figsize=(6 * len(runs), 5))
axes = np.atleast_1d(axes)
for ax, (run_no, data) in zip(axes, runs.items()):
    gmd = data.gmd.ravel()
    s   = data.vls_sums.ravel()
    ok  = np.isfinite(gmd) & np.isfinite(s)
    x, y = gmd[ok], s[ok]

    edges = np.percentile(x, np.linspace(0, 100, N_GMD_BINS + 1))
    _, bool_ar = arb_bool_ar(edges, x)
    my, sy, _ = bin_and_average(bool_ar, y)
    mx, _,  _ = bin_and_average(bool_ar, x)

    slope, intercept = np.polyfit(x, y, 1)
    r = float(np.corrcoef(x, y)[0, 1])
    xline = np.linspace(mx.min(), mx.max(), 100)

    h = ax.hist2d(x, y,
                  bins=[edges, np.linspace(0, np.percentile(y, 95), 51)],
                  cmap="viridis", cmin=1, norm=mcolors.LogNorm())
    fig.colorbar(h[3], ax=ax, label="Shots per bin")
    ax.errorbar(mx, my, yerr=sy, fmt="o-", color="white",
                capsize=3, lw=1.2)
    ax.plot(xline, slope * xline + intercept,
            color="white", ls="--", lw=1.2,
            label=f"slope={slope:.3g}, r={r:.3f}")
    ax.set_xlabel("GMD (uJ)")
    ax.set_ylabel("Per-shot VLS sum (arb.)")
    ax.set_title(f"run {run_no}")
    ax.legend(loc="upper left", framealpha=0.85)

fig.suptitle("GMD vs VLS sum")
fig.tight_layout()
plt.show()

## Average VLS spectrum per GMD bin

One panel per run; lines coloured by mean GMD per percentile bin.

In [ ]:
N_GMD_BINS = 10

fig, axes = plt.subplots(1, len(runs),
                         figsize=(6 * len(runs), 5), sharey=True)
axes = np.atleast_1d(axes)
for ax, (run_no, data) in zip(axes, runs.items()):
    gmd = data.gmd.ravel()
    s   = data.vls_sums.ravel()
    ok  = np.isfinite(gmd) & np.isfinite(s)

    edges = np.percentile(gmd[ok], np.linspace(0, 100, N_GMD_BINS + 1))
    _, bool_ar = arb_bool_ar(edges, gmd[ok])
    mean_gmd, _, _ = bin_and_average(bool_ar, gmd[ok])

    specs = data.vls.reshape(-1, data.vls.shape[-1])[ok]
    mean_spec, _, _ = bin_and_average(bool_ar, specs)

    norm = Normalize(vmin=float(mean_gmd[0]), vmax=float(mean_gmd[-1]))
    for cent, spec in zip(mean_gmd, mean_spec):
        ax.plot(data.vls_pixels, spec,
                color=plt.cm.viridis(norm(cent)), lw=1.2)
    sm = ScalarMappable(norm=norm, cmap="viridis"); sm.set_array([])
    fig.colorbar(sm, ax=ax, label="Mean GMD per bin (uJ)")
    ax.set_xlabel("Pixel")
    ax.set_ylabel("Mean intensity (arb.)")
    ax.set_title(f"run {run_no}  ({N_GMD_BINS} percentile GMD bins)")
    ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

## Average VLS spectrum per COM bin

COM bin edges are pooled across runs so the two panels share the
same colourbar.

In [ ]:
N_COM_BINS = 10

com_pool = np.concatenate([d.vls_coms.ravel() for d in runs.values()])
com_pool = com_pool[np.isfinite(com_pool)]
com_edges = np.percentile(com_pool, np.linspace(5, 95, N_COM_BINS + 1))
norm_com  = Normalize(vmin=float(com_edges[0]), vmax=float(com_edges[-1]))

fig, axes = plt.subplots(1, len(runs),
                         figsize=(6 * len(runs), 5), sharey=True)
axes = np.atleast_1d(axes)
for ax, (run_no, data) in zip(axes, runs.items()):
    com = data.vls_coms.ravel()
    ok  = np.isfinite(com)

    _, bool_ar = arb_bool_ar(com_edges, com[ok])
    specs = data.vls.reshape(-1, data.vls.shape[-1])[ok]
    mean_spec, _, _ = bin_and_average(bool_ar, specs)
    mean_com,  _, _ = bin_and_average(bool_ar, com[ok])

    for cent, spec in zip(mean_com, mean_spec):
        ax.plot(data.vls_pixels, spec,
                color=plt.cm.plasma(norm_com(cent)), lw=1.2)
    sm = ScalarMappable(norm=norm_com, cmap="plasma"); sm.set_array([])
    fig.colorbar(sm, ax=ax, label="Mean COM per bin (pixel)")
    ax.set_xlabel("Pixel")
    ax.set_ylabel("Mean intensity (arb.)")
    ax.set_title(f"run {run_no}  ({N_COM_BINS} percentile COM bins)")
    ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

## Final comparison: GMD-normalised spectra per (COM bin x GMD bin)

For each (COM bin x GMD bin) pair, compute

    sum(spectra) / sum(GMD)

via `bin_and_sum_ratio` so the normalisation is the energy-weighted
average (per CLAUDE.md convention) rather than `mean(spec / gmd)`.

One panel per GMD bin; within each panel, the COM bins are coloured
by their centre and the runs are distinguished by line style (solid
= first run, dotted = second). COM and GMD edges are pooled across
the runs so the binning is shared.

In [ ]:
N_COM_BINS = 10
N_GMD_BINS = 5

com_pool = np.concatenate([d.vls_coms.ravel() for d in runs.values()])
com_edges = np.percentile(com_pool[np.isfinite(com_pool)],
                          np.linspace(5, 95, N_COM_BINS + 1))

gmd_pool = np.concatenate([d.gmd.ravel() for d in runs.values()])
gmd_edges = np.percentile(gmd_pool[np.isfinite(gmd_pool)],
                          np.linspace(5, 95, N_GMD_BINS + 1))


def per_bin_normalised(data, com_edges, gmd_edges):
    """Return sum(spec)/sum(gmd) per (GMD bin, COM bin), shape (n_gmd, n_com, n_pix)."""
    com   = data.vls_coms.ravel()
    gmd   = data.gmd.ravel()
    specs = data.vls.reshape(-1, data.vls.shape[-1])
    ok    = np.isfinite(com) & np.isfinite(gmd)
    com, gmd, specs = com[ok], gmd[ok], specs[ok]

    com_cents, com_mask = arb_bool_ar(com_edges, com)
    gmd_cents, gmd_mask = arb_bool_ar(gmd_edges, gmd)

    n_com = com_mask.shape[0]
    n_gmd = gmd_mask.shape[0]
    out = np.full((n_gmd, n_com, specs.shape[1]), np.nan)
    for g in range(n_gmd):
        joint = com_mask & gmd_mask[g][None, :]    # (n_com, n_shots)
        ratio, _ = bin_and_sum_ratio(joint, specs, gmd)
        out[g] = ratio
    return out, com_cents, gmd_cents


matrices  = {r: per_bin_normalised(d, com_edges, gmd_edges)
             for r, d in runs.items()}
com_cents = next(iter(matrices.values()))[1]
gmd_cents = next(iter(matrices.values()))[2]
styles    = ["-", ":", "--", "-."]
norm_com  = Normalize(vmin=float(com_cents[0]), vmax=float(com_cents[-1]))
pixel_ax  = next(iter(runs.values())).vls_pixels

for g in range(N_GMD_BINS):
    fig, ax = plt.subplots(figsize=(11, 5))
    for (run_no, (M, *_)), style in zip(matrices.items(), styles):
        for cent, spec in zip(com_cents, M[g]):
            ax.plot(pixel_ax, spec,
                    color=plt.cm.plasma(norm_com(cent)),
                    lw=1.2, ls=style)
        ax.plot([], [], color="black", lw=1.2, ls=style,
                label=f"run {run_no}")
    sm = ScalarMappable(norm=norm_com, cmap="plasma"); sm.set_array([])
    fig.colorbar(sm, ax=ax, label="COM bin centre (pixel)")
    ax.set_xlabel("Pixel")
    ax.set_ylabel("sum(spectra) / sum(GMD)")
    ax.set_title(f"GMD bin {g}: [{gmd_edges[g]:.2f}, {gmd_edges[g+1]:.2f}] uJ"
                 f"  (centre={gmd_cents[g]:.2f})")
    ax.grid(alpha=0.3)
    ax.legend(loc="best")
    fig.tight_layout()
    plt.show()

## Average |FFT| comparison

Fourier-transform each per-shot spectrum along the pixel axis, then
average the magnitude across all shots in the run. Interference
fringes embedded in individual spectra show up as peaks here:
their phase varies shot-to-shot, so they wash out of the mean
spectrum but survive in `<|FFT|>`.

`np.fft.rfft` keeps only positive frequencies; the unit is
cycles / pixel.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for run_no, data in runs.items():
    flat = data.vls.reshape(-1, data.vls.shape[-1])
    ok = np.all(np.isfinite(flat), axis=1)
    spec = flat[ok]
    n_pix = spec.shape[1]
    fft_abs = np.abs(np.fft.rfft(spec, axis=1))
    mean_fft = np.nanmean(fft_abs, axis=0)
    freqs = np.fft.rfftfreq(n_pix, d=1.0)
    # Skip the DC bin so the y-scale is dominated by the structure of interest.
    ax.semilogy(freqs[1:], mean_fft[1:], lw=1.4,
                label=f"run {run_no}  ({spec.shape[0]} shots)")

ax.set_xlabel("Frequency (cycles / pixel)")
ax.set_ylabel("<|FFT(spectrum)|>  (arb.)")
ax.set_title("Mean |FFT| of per-shot spectra  (DC bin skipped)")
ax.grid(alpha=0.3, which="both")
ax.legend()
fig.tight_layout()
plt.show()